In [1]:
import CDutils
import scipy.io as sio
from sklearn.preprocessing import RobustScaler
import numpy as np
from ts2vec import TS2Vec
from scipy.io import savemat
import ruptures as rpt
import os
from sklearn.metrics import f1_score

In [2]:
path = r"D:\Code_Hansong\Contraction detection manuscript\Contaction_detection\Data_sorted"
dir_list = os.listdir(path)

In [5]:
len(dir_list)

12

In [6]:
def contraction_detection_single_ch(sig,features,p):
    
    algo = rpt.KernelCPD(kernel="cosine").fit(features)
    result = algo.predict(n_bkps=p)
   
   
    c = np.zeros(sig.shape[0])
    result.insert(0, 0)
    
    first_sub = sig[result[0]:result[1]]
    ef = np.sum(np.square(first_sub))/first_sub.shape[0]
    second_sub =  sig[result[1]:result[2]]
    es = np.sum(np.square(second_sub))/second_sub.shape[0]

    last_sub = sig[result[-2]:result[-1]]
    el = np.sum(np.square(last_sub))/last_sub.shape[0]
    second_last_sub = sig[result[-3]:result[-2]]
    e_sl = np.sum(np.square(second_last_sub))/second_last_sub.shape[0]
    
    # print(result)
    if ef>es:  
        c[result[0]:result[1]]=1
        
    if el>e_sl:
        c[result[-2]:result[-1]]=1
        
    for j in range(len(result)-3):
        sub1 = sig[result[j]:result[j+1]]
        sub2 = sig[result[j+1]:result[j+2]]
        sub3 = sig[result[j+2]:result[j+3]]
        e1 = np.sum(np.square(sub1))/sub1.shape[0]
        e2 = np.sum(np.square(sub2))/sub2.shape[0]
        e3 = np.sum(np.square(sub3))/sub3.shape[0]
        if e2>e1 and e2>e3:
            c[result[j+1]:result[j+2]]=1
        
    return c

In [7]:
f1_list = []
cp_score_list = []
cpn_list = []

for pt in range(len(dir_list)):
    data = sio.loadmat(path + '/' + dir_list[pt])
    modulation_ds = data['tmp_y']
    label_ds = data['newClusterSig']
    connectivity = data['Uterus']['tri'][0][0]
    conn = connectivity.astype(int)-1

    ch_num = modulation_ds.shape[0]
    conn_num = conn.shape[0]
    signal_length = modulation_ds.shape[1]
    
    modulation_ar_removal = CDutils.ar_remove_patient(modulation_ds)
    profile = CDutils.matrix_profile(modulation_ar_removal,40)
    cp_num = CDutils.change_point_num(profile,t=0.49,d=200)
    
    EMG_sig_t = np.transpose(modulation_ds)
    transformer = RobustScaler(with_centering=True, with_scaling=True, quantile_range=(15,85),).fit(EMG_sig_t)
    EMG_scaled = transformer.transform(EMG_sig_t)

    train_data = EMG_scaled.reshape(EMG_scaled.shape[0],EMG_scaled.shape[1],1)
    print(train_data.shape)

    model = TS2Vec(
    input_dims=1,
    device=0,
    output_dims=4
    )
    loss_log = model.fit(
    train_data,
    verbose=True
    )
    train_repr = model.encode(train_data)
    print('representation shape:' + str(train_repr.shape))

    
    c_detection = np.zeros_like(modulation_ds)

    for i in range(ch_num): 
        sig = EMG_scaled[:,i]
    
        features = train_repr[:,i,:]
        
        features_reshape = np.squeeze(features)
        
        c_detection[i,:] = contraction_detection_single_ch(sig,features_reshape,cp_num)
        
    f1_array = np.zeros((ch_num,))
    cp_score_array = np.zeros(ch_num,)
    cpn_array = np.zeros(ch_num,)
    for ch in range(ch_num):
        pre_cpds = CDutils.find_cpd(c_detection[ch,:], 1)
        gt_cpds = CDutils.find_cpd(label_ds[ch,:], 1)
        f1_array[ch] = f1_score(label_ds[ch,:], c_detection[ch,:],zero_division=1.0)
        cp_score_array[ch] = CDutils.cp_score(gt_cpds,pre_cpds,signal_length)
        cpn_array[ch] = (np.abs(len(gt_cpds)-len(pre_cpds)))/len(gt_cpds)
    print(np.median(f1_array))
    print(np.median(cp_score_array))
    print(np.median(cpn_array))
    f1_list.append(np.median(f1_array))
    cp_score_list.append(np.median(cp_score_array))
    cpn_list.append(np.median(cpn_array))

D:\Code_Hansong\ts2vec-main\CDutils.py:174: RuntimeWarning: invalid value encountered in scalar divide
  en_masked = np.sum(np.square(sig_masked)) / (sig_masked.shape[0])
D:\Code_Hansong\ts2vec-main\CDutils.py:203: RuntimeWarning: invalid value encountered in scalar divide
  en_pre = np.sum(np.square(pre)) / (pre.shape[0])


(2311, 320, 1)


C:\Users\jmeng\miniconda3\envs\ts2\Lib\site-packages\torch\nn\modules\conv.py:306: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\cudnn\Conv_v8.cpp:919.)
  return F.conv1d(input, weight, bias, self.stride,
C:\Users\jmeng\miniconda3\envs\ts2\Lib\site-packages\torch\autograd\graph.py:744: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\cudnn\Conv_v8.cpp:919.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch #0: loss=3.1223170459270477
Epoch #1: loss=2.721166009704272
Epoch #2: loss=2.405322827398777
Epoch #3: loss=2.384857025411394
representation shape:(2311, 320, 4)
0.8549862188830202
0.006887350353382374
0.0
(2335, 320, 1)


C:\Users\jmeng\miniconda3\envs\ts2\Lib\site-packages\torch\autograd\graph.py:744: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\cudnn\Conv_v8.cpp:919.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
C:\Users\jmeng\miniconda3\envs\ts2\Lib\site-packages\torch\nn\modules\conv.py:306: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\cudnn\Conv_v8.cpp:919.)
  return F.conv1d(input, weight, bias, self.stride,


...

(1951, 320, 1)
Epoch #0: loss=3.4223092508710122
Epoch #1: loss=3.276342644179163
Epoch #2: loss=3.1605421826859152
Epoch #3: loss=3.070240278874547
representation shape:(1951, 320, 4)
0.375886729770574
0.010622757560225526
0.21428571428571427


D:\Code_Hansong\ts2vec-main\CDutils.py:174: RuntimeWarning: invalid value encountered in scalar divide
  en_masked = np.sum(np.square(sig_masked)) / (sig_masked.shape[0])
D:\Code_Hansong\ts2vec-main\CDutils.py:203: RuntimeWarning: invalid value encountered in scalar divide
  en_pre = np.sum(np.square(pre)) / (pre.shape[0])


(2194, 322, 1)


C:\Users\jmeng\miniconda3\envs\ts2\Lib\site-packages\torch\nn\modules\conv.py:306: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\cudnn\Conv_v8.cpp:919.)
  return F.conv1d(input, weight, bias, self.stride,
C:\Users\jmeng\miniconda3\envs\ts2\Lib\site-packages\torch\autograd\graph.py:744: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\cudnn\Conv_v8.cpp:919.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch #0: loss=3.186598196516942
Epoch #1: loss=2.7156733431085183
Epoch #2: loss=2.5306193689360237
Epoch #3: loss=2.454828455500359
representation shape:(2194, 322, 4)
0.6418496003843861
0.018103781663489957
0.3


In [8]:
mdic = {"f1_list": f1_list,"cp_score_list": cp_score_list,"cpn_list": cpn_list}
savemat(r"D:\Code_Hansong\Contraction detection manuscript\Contaction_detection\results\ts2vec_result1023.mat", mdic)